# Проект моё решающее дерево
В этот раз я возьму хороший для обучения датасет, чтобы реализовать с нуля своё решаюшее дерево

## 0. Импорты

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

from kagglehub import KaggleDatasetAdapter
from sklearn.preprocessing import LabelEncoder

## 1. Про датасет
Информация о космических объектах, нужно определить класс объекта

In [30]:
df = pd.read_csv('sdss19_imbalanced.csv')

## 2. EDA

In [31]:
display(df.head(5))
display(df.describe())
display(df.info())
display(df.isnull().sum())

,obj_class,ra,dec,u,g,r,i,zband,petroR50_r,extinction_r,pmRa,pmDec,pmRaErr,pmDecErr
0,STAR,94.151185,0.556955,18.55815,17.13832,16.46642,16.17820,16.00801,0.616056,0.971888,2.823026,2.655500,3.005414,3.005414
1,STAR,94.162827,0.595549,18.19262,17.05079,16.95350,16.97508,16.99701,0.633574,0.982412,4.386912,2.636851,3.002324,3.002324
2,STAR,94.193505,0.545911,17.16965,15.73020,15.58151,15.56168,15.50965,0.631930,0.976873,1.592040,-2.781029,2.939655,2.939655
3,STAR,94.243501,0.447088,17.21898,16.08286,16.01093,16.03192,16.02211,0.613530,0.955022,2.372745,1.497425,2.941542,2.941542
4,STAR,94.312293,0.487440,16.75434,15.33976,15.09668,15.02321,14.91764,0.634270,0.975247,1.869048,2.719901,2.939654,2.939654


,ra,dec,u,g,r,i,zband,petroR50_r,extinction_r,pmRa,pmDec,pmRaErr,pmDecErr
count,280002.000000,280002.000000,280002.000000,280002.000000,280002.000000,280002.000000,280002.000000,280002.000000,280002.000000,280002.000000,280002.000000,280002.000000,280002.000000
mean,165.950152,22.730146,21.432372,19.845792,18.885888,18.386532,18.106354,1.503447,0.098359,-0.507517,-0.912034,2.771366,2.771366
std,66.259977,27.135085,2.267683,1.931811,1.712333,1.625857,1.641316,1.664955,0.191333,8.154989,8.353731,2.540176,2.540176
min,0.004289,-11.244273,11.960910,11.840130,11.629170,11.051390,10.616260,0.115081,0.008580,-56.995480,-56.995620,0.000000,0.000000
25%,129.205036,-0.242879,19.665615,18.251315,17.465460,17.063038,16.780700,0.709724,0.051280,-2.373086,-2.837724,0.000000,0.000000
50%,163.284179,3.078136,21.239290,20.030490,19.097320,18.618320,18.281125,1.229612,0.077344,0.000000,0.000000,3.024227,3.024227
75%,206.555414,51.996310,23.071347,21.534945,20.418405,19.647187,19.256105,1.872871,0.111708,1.687489,1.344996,3.879420,3.879420
max,359.997379,68.739531,31.143020,27.851710,30.037920,30.028860,28.234510,223.407400,34.052850,56.858720,56.941540,191.487700,191.487700


<class 'pandas.DataFrame'>
RangeIndex: 280002 entries, 0 to 280001
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   obj_class     280002 non-null  str    
 1   ra            280002 non-null  float64
 2   dec           280002 non-null  float64
 3   u             280002 non-null  float64
 4   g             280002 non-null  float64
 5   r             280002 non-null  float64
 6   i             280002 non-null  float64
 7   zband         280002 non-null  float64
 8   petroR50_r    280002 non-null  float64
 9   extinction_r  280002 non-null  float64
 10  pmRa          280002 non-null  float64
 11  pmDec         280002 non-null  float64
 12  pmRaErr       280002 non-null  float64
 13  pmDecErr      280002 non-null  float64
dtypes: float64(13), str(1)
memory usage: 31.3 MB


None

obj_class       0
ra              0
dec             0
u               0
g               0
r               0
i               0
zband           0
petroR50_r      0
extinction_r    0
pmRa            0
pmDec           0
pmRaErr         0
pmDecErr        0
dtype: int64

In [33]:
X = df.drop(columns='obj_class')
y = pd.Series(df['obj_class'])

display(X)
display(y)

,ra,dec,u,g,r,i,zband,petroR50_r,extinction_r,pmRa,pmDec,pmRaErr,pmDecErr
0,94.151185,0.556955,18.55815,17.13832,16.46642,16.17820,16.00801,0.616056,0.971888,2.823026,2.655500,3.005414,3.005414
1,94.162827,0.595549,18.19262,17.05079,16.95350,16.97508,16.99701,0.633574,0.982412,4.386912,2.636851,3.002324,3.002324
2,94.193505,0.545911,17.16965,15.73020,15.58151,15.56168,15.50965,0.631930,0.976873,1.592040,-2.781029,2.939655,2.939655
3,94.243501,0.447088,17.21898,16.08286,16.01093,16.03192,16.02211,0.613530,0.955022,2.372745,1.497425,2.941542,2.941542
4,94.312293,0.487440,16.75434,15.33976,15.09668,15.02321,14.91764,0.634270,0.975247,1.869048,2.719901,2.939654,2.939654
...,...,...,...,...,...,...,...,...,...,...,...,...,...
279997,205.635560,3.082127,17.44481,15.99625,15.47226,15.19528,14.98716,5.335046,0.055828,24.441260,5.196535,2.561155,2.561155
279998,205.677580,3.039798,20.10751,19.15912,18.83749,18.71170,18.65036,0.629756,0.054367,-4.931767,-3.462556,3.518237,3.518237
279999,205.765360,3.031234,20.54722,18.40636,17.20650,16.69568,16.28923,1.972270,0.055669,2.367301,2.342535,2.751652,2.751652
280000,205.655588,3.064547,22.36661,20.88627,19.19122,18.54666,18.15465,1.500211,0.054942,-7.516692,-5.082550,4.913247,4.913247


0           STAR
1           STAR
2           STAR
3           STAR
4           STAR
           ...  
279997    GALAXY
279998      STAR
279999    GALAXY
280000    GALAXY
280001    GALAXY
Name: obj_class, Length: 280002, dtype: str

## 3. Feature engeeniring
Пока ничего не придумал

## 4. Реализация класса

In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, left_child=None, right_child=None, value=None, depth=None, impurity=None, n_samples=None):
        self.feature = feature
        self.threshold = threshold
        self.left_child = left_child
        self.right_child = right_child
        self.value = value
        self.depth = depth
        self.impurity = impurity

class MyDecisionTreeClassifier:
    def __init__(self, max_depth = 10, min_samples_split = 5, min_samples_leaf = 3, max_leaf_nodes = 20, criterion = 'gini'):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_leaf_nodes = max_leaf_nodes
        self.criterion = {
            'gini': lambda p: 1 - np.sum(p**2),
            'entropy': lambda p: -np.sum(p * np.log2(p + 1e-9))
        }[criterion]
    
    def build_tree(self, X, y, depth, n_samples, impurity):
        unique, counts = np.unique(y, return_counts=True)
        if(depth > self.max_depth or n_samples < self.min_samples_split or n_samples == np.max(counts)):
            return Node(value=np.argmax(counts))
        else:
            best_feature = None
            best_threshold = None
            best_impurity = 0
            best_impurity_left = 0
            best_impurity_right = 0
            best_X_left = []
            best_y_left = []
            best_X_right = []
            best_y_right = []
            for feature in X.columns:
                for i, threshold in enumerate(np.unique(X[feature])):
                    left_mask = X[feature] <= threshold
                    right_mask = X[feature] > threshold
                    
                    X_left = X[left_mask]
                    y_left = y[left_mask]
                    X_right = X[right_mask]
                    y_right = y[right_mask]
                    
                    unique_left, counts_left = np.unique(y_left, return_counts=True)
                    unique_right, counts_right = np.unique(y_right, return_counts=True)
                    
                    p_left = np.zeros(len(self.classes))
                    p_right = np.zeros(len(self.classes))
                    
                    for class_, counts in zip(unique_left, counts_left):
                        p_left[class_] = counts / len(y_left)
                        
                    for class_, counts in zip(unique_right, counts_right):
                        p_right[class_] = counts / len(y_right)
                        
                    impurity_left = self.criterion(p_left)
                    impurity_right = self.criterion(p_right)
                    
                    new_impurity = len(X) * impurity - len(X_left) * impurity_left - len(X_right) * impurity_right
                    
                    if(new_impurity > best_impurity):
                        best_impurity = new_impurity
                        best_impurity_left = impurity_left
                        best_impurity_right = impurity_right
                        best_feature = feature
                        best_threshold = threshold
                        best_X_left = X_left
                        best_y_left = y_left
                        best_X_right = X_right
                        best_y_right = y_right
            
            
            if(best_feature == None):
                return Node(value=np.argmax(counts))
            
            else:            
                left_child = self.build_tree(best_X_left, best_y_left, depth=depth+1, n_samples=len(best_y_left), impurity=best_impurity_left)
                right_child = self.build_tree(best_X_right, best_y_right, depth=depth+1, n_samples=len(best_y_right), impurity=best_impurity_right)
                return Node(feature=best_feature, threshold=best_threshold, left_child=left_child, right_child=right_child, depth=depth, impurity=best_impurity, n_samples=n_samples)
    
    def fit(self, X, y):
        self.classes, self.counts = np.unique(y, return_counts=True)
        self.features = X.columns
        
        p = np.zeros(len(self.classes))
        
        for class_, counts in zip(self.classes, self.counts):
            p[class_] = counts / len(y)
            
        impurity = self.criterion(p)
        
        self.root = self.build_tree(X, y, depth=0, n_samples=len(y), impurity=impurity)